# Prompt Engineering - Hands-On Tour

A single notebook walking through the core techniques from `../docs/`:
zero-shot vs few-shot &rarr; chain-of-thought &rarr; structured output &rarr;
context injection &rarr; prompt chaining &rarr; evaluation. Same framing as
module 01's notebook - every section ties back to something I'd
actually reach for as a platform/DevOps engineer.

## Setup

```bash
ollama pull llama3.1:8b
pip install openai jsonschema
```

Runs against a local Ollama model - no API key, no cost.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "llama3.1:8b"

## 1. Zero-shot vs. few-shot

See `docs/03-Zero-Shot-vs-Few-Shot.md`. Few-shot examples calibrate the
model to a convention it can't know from training alone - here,
"replica disk usage is MEDIUM, not HIGH."

In [ ]:
alert = "Disk usage at 92% on db-replica-02, 20 minutes to full"

zero_shot = f'Classify severity as LOW, MEDIUM, or HIGH:\n"{alert}"\nSeverity:'
few_shot = f'''Classify severity as LOW, MEDIUM, or HIGH.

Alert: "Disk usage at 98% on db-primary-01, 5 minutes to full"
Severity: HIGH

Alert: "Disk usage at 90% on db-replica-01, 30 minutes to full"
Severity: MEDIUM

Alert: "{alert}"
Severity:'''

for label, prompt in [("zero-shot", zero_shot), ("few-shot", few_shot)]:
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    print(f"{label}: {r.choices[0].message.content.strip()}")

## 2. Chain-of-thought

See `docs/04-Chain-of-Thought.md`. Forcing intermediate steps onto the
page measurably helps on multi-step problems - worked answer: 5
batches x 60s + 30s readiness delay = 330s.

In [ ]:
problem = ("A rolling deploy replaces 10 pods, 2 at a time, waiting 60s "
           "between batches, with a 30s readiness probe delay before the "
           "first batch counts as healthy. How many seconds minimum "
           "before all 10 pods are updated?")

direct = f"{problem} Answer with just the number."
cot = f"{problem} Think step by step, then answer as 'Answer: N seconds'."

for label, prompt in [("direct", direct), ("chain-of-thought", cot)]:
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    print(f"--- {label} ---\n{r.choices[0].message.content}\n")

## 3. Structured output, validated

See `docs/06-Structured-Output-Prompting.md`. Explicit instruction +
JSON mode + schema validation - never trust `json.loads()` succeeding
alone.

In [ ]:
import json
from jsonschema import validate

schema = {
    "type": "object",
    "properties": {"host": {"type": "string"}, "value": {"type": "number"}},
    "required": ["host", "value"],
}

prompt = 'Extract as JSON with keys "host" and "value" (number): "Disk usage at 92% on db-primary-02"'
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"},
    temperature=0,
)
parsed = json.loads(r.choices[0].message.content)
validate(instance=parsed, schema=schema)
print(parsed)

## 4. Context injection, delimited

See `docs/08-Context-Injection.md`. Compare an ungrounded guess
against an answer grounded in real, clearly-delimited policy text.

In [ ]:
policy = "Per SRE-042: disk >90% on a PRIMARY db is HIGH severity; on a REPLICA it's MEDIUM."
question = "Is 92% disk usage on a replica database node HIGH or MEDIUM severity?"

ungrounded = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}], temperature=0)
print("ungrounded:", ungrounded.choices[0].message.content[:150])

grounded_prompt = f"Answer using ONLY the info in <context>.\n<context>\n{policy}\n</context>\nQuestion: {question}"
grounded = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": grounded_prompt}], temperature=0)
print("grounded:  ", grounded.choices[0].message.content[:150])

## 5. Prompt chaining

See `docs/12-Prompt-Chaining.md`. Break "analyze this incident" into
independently-checkable steps instead of one prompt doing everything.

In [ ]:
incident_log = '''
09:01 UTC - Alert fired: high 5xx rate on checkout-service
09:06 UTC - kubectl describe showed OOMKilled, exit code 137
09:08 UTC - Recent deploy bumped batch size in worker config
09:10 UTC - Rolled back to previous deployment revision
09:13 UTC - 5xx rate returned to baseline
'''

root_cause = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"In one sentence, what is the root cause here?\n{incident_log}"}],
    temperature=0,
).choices[0].message.content.strip()
print("root cause:", root_cause)

status_update = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Write a 2-sentence customer-facing update, no jargon: {root_cause}"}],
    temperature=0.3,
).choices[0].message.content.strip()
print("status update:", status_update)

## 6. Evaluating prompt variants

See `docs/16-Evaluating-Prompt-Quality.md`. "v2 is better" should be a
number, not a feeling - score both variants against a small labeled
set that includes the tricky replica case.

In [ ]:
eval_set = [
    {"input": "CPU at 45% on web-node-01", "expected": "LOW"},
    {"input": "Disk at 98% on db-primary-01, 5 min to full", "expected": "HIGH"},
    {"input": "Disk at 92% on a REPLICA db node", "expected": "MEDIUM"},
]

v1 = "Classify severity as LOW, MEDIUM, or HIGH: {alert}"
v2 = "You are an SRE. Disk >90% on PRIMARY is HIGH, on REPLICA is MEDIUM. Classify: {alert}"

def score(template):
    correct = 0
    for case in eval_set:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": template.format(alert=case["input"])}],
            temperature=0,
        )
        if case["expected"] in r.choices[0].message.content.upper():
            correct += 1
    return correct / len(eval_set)

print(f"v1 accuracy: {score(v1):.0%}")
print(f"v2 accuracy: {score(v2):.0%}")

## Wrap-up

| Section | Concept | Docs chapter |
| --- | --- | --- |
| 1 | Zero-shot vs. few-shot | 03 |
| 2 | Chain-of-thought | 04 |
| 3 | Structured output | 06 |
| 4 | Context injection | 08 |
| 5 | Prompt chaining | 12 |
| 6 | Evaluation | 16 |

Not covered here but worth running from `../examples/`:
`08_tool_calling.py` (function calling) and `09_prompt_injection_demo.py`
(the attack this whole module warns about, and its mitigation).

Next: `docs/17-Prompt-Versioning.md` and `docs/18-Best-Practices.md` for
how these techniques get wired into something safe to ship.